## 1. Initialize Project Environment
Import libraries for enrichment analysis and biological interpretation.

In [4]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Set

import numpy as np
import pandas as pd
from scipy import stats

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)

pandas 2.2.3
numpy 2.1.3


## 2. Define Configuration Parameters
Centralize enrichment analysis settings.

In [5]:
@dataclass
class EnrichmentConfig:
    modules_file: Path = Path("artifacts/task3_module_assignments.csv")
    hub_file: Path = Path("artifacts/task3_hub_genes.csv")
    true_modules_file: Path = Path("artifacts/task1_true_modules.csv")
    export_dir: Path = Path("artifacts")
    pvalue_threshold: float = 0.05
    min_genes_for_enrichment: int = 5

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        for k in ["modules_file", "hub_file", "true_modules_file", "export_dir"]:
            info[k] = str(info[k])
        return info


CONFIG = EnrichmentConfig()
CONFIG.describe()

{'modules_file': 'artifacts/task3_module_assignments.csv',
 'hub_file': 'artifacts/task3_hub_genes.csv',
 'true_modules_file': 'artifacts/task1_true_modules.csv',
 'export_dir': 'artifacts',
 'pvalue_threshold': 0.05,
 'min_genes_for_enrichment': 5}

## 3. Load Module Data
Load detected modules and hub gene information.

In [6]:
# Load data
modules = pd.read_csv(CONFIG.modules_file)
hubs = pd.read_csv(CONFIG.hub_file)
true_modules = pd.read_csv(CONFIG.true_modules_file)

logging.info(f"Loaded {len(modules)} genes with module assignments")
logging.info(f"Hub genes: {hubs['IsHub'].sum()}")

# Merge with true module information
merged = modules.merge(true_modules, on="Gene", how="left")
merged = merged.merge(
    hubs[["Gene", "IsHub", "Connectivity", "IntramodularConnectivity"]],
    on="Gene",
    how="left",
)
merged.head(10)

[INFO] Loaded 67 genes with module assignments
[INFO] Hub genes: 7


,Gene,Module,TrueModule,IsHub,Connectivity,IntramodularConnectivity
0,BIRC5,1,TP53_pathway,False,22.455483,13.223058
1,APAF1,1,TP53_pathway,True,24.394018,14.398655
2,HTRA2,1,TP53_pathway,False,23.133109,13.566186
3,CASP9,1,TP53_pathway,False,20.961068,12.775651
4,MDM2,1,TP53_pathway,False,25.040650,14.343693
5,BAX,1,TP53_pathway,False,21.123719,12.466298
6,CASP3,1,TP53_pathway,False,20.309391,12.031795
7,AIF,1,TP53_pathway,False,22.421817,13.732422
8,XIAP,1,TP53_pathway,False,24.316474,13.472726
9,MCL1,1,TP53_pathway,False,25.676963,14.212080


## 4. Define Pathway Gene Sets
Create gene sets for key biological pathways related to TP53 function.

In [7]:
# Define curated pathway gene sets (based on KEGG/GO annotations)
PATHWAY_GENE_SETS = {
    "Apoptosis": {
        "TP53",
        "BAX",
        "PUMA",
        "NOXA",
        "BID",
        "APAF1",
        "CASP9",
        "CASP3",
        "BCL2",
        "BCL2L1",
        "MCL1",
        "BIRC5",
        "XIAP",
        "CYCS",
        "DIABLO",
        "ENDOG",
        "AIF",
        "HTRA2",
    },
    "Cell_Cycle": {
        "CDKN1A",
        "CDKN2A",
        "RB1",
        "E2F1",
        "E2F2",
        "CCND1",
        "CCNE1",
        "CDK2",
        "CDK4",
        "CDK6",
        "CCNB1",
        "CDC25A",
        "CDC25C",
        "PLK1",
        "AURKA",
        "AURKB",
        "BUB1",
        "MAD2L1",
        "CHEK1",
        "CHEK2",
    },
    "DNA_Damage_Response": {
        "ATM",
        "ATR",
        "BRCA1",
        "BRCA2",
        "RAD51",
        "XRCC1",
        "PARP1",
        "PARP2",
        "MLH1",
        "MSH2",
        "MSH6",
        "OGG1",
        "XPA",
        "XPC",
        "ERCC1",
        "ERCC2",
        "POLE",
        "POLD1",
        "FEN1",
        "LIG1",
        "LIG3",
    },
    "p53_Signaling": {
        "TP53",
        "MDM2",
        "CDKN1A",
        "BAX",
        "PUMA",
        "NOXA",
        "GADD45A",
        "SFN",
        "SERPINE1",
        "THBS1",
        "TSC2",
        "SESN1",
        "SESN2",
        "TIGAR",
        "SCO2",
    },
    "Metabolism": {
        "HIF1A",
        "LDHA",
        "PKM",
        "GLUT1",
        "HK2",
        "PFKFB3",
        "SCO2",
        "TIGAR",
        "GLS2",
        "PTEN",
        "AKT1",
        "MTOR",
        "AMPK",
        "PGC1A",
        "SIRT1",
        "SIRT3",
        "UCP2",
        "CPT1A",
        "ACACA",
        "FASN",
    },
}

# Show pathway sizes
pathway_summary = {name: len(genes) for name, genes in PATHWAY_GENE_SETS.items()}
pd.DataFrame([pathway_summary]).T.rename(columns={0: "Gene_Count"})

,Gene_Count
Apoptosis,18
Cell_Cycle,20
DNA_Damage_Response,21
p53_Signaling,15
Metabolism,20


## 5. Perform Enrichment Analysis
Use Fisher's exact test to assess pathway enrichment in each module.

In [8]:
def fisher_enrichment(
    module_genes: Set[str], pathway_genes: Set[str], background_genes: Set[str]
) -> Dict:
    """
    Perform Fisher's exact test for pathway enrichment.

    Returns dict with:
    - overlap: genes in both module and pathway
    - odds_ratio: enrichment score
    - pvalue: significance
    """
    # Ensure all sets are within background
    module_genes = module_genes & background_genes
    pathway_genes = pathway_genes & background_genes

    # Contingency table
    a = len(module_genes & pathway_genes)  # In module AND in pathway
    b = len(module_genes - pathway_genes)  # In module but NOT in pathway
    c = len(pathway_genes - module_genes)  # NOT in module but in pathway
    d = len(background_genes - module_genes - pathway_genes)  # Neither

    # Fisher's exact test
    contingency = [[a, b], [c, d]]
    odds_ratio, pvalue = stats.fisher_exact(contingency, alternative="greater")

    return {
        "overlap_count": a,
        "overlap_genes": module_genes & pathway_genes,
        "module_size": len(module_genes),
        "pathway_size": len(pathway_genes),
        "odds_ratio": odds_ratio,
        "pvalue": pvalue,
    }


def run_enrichment_analysis(
    modules_df: pd.DataFrame, pathway_sets: Dict[str, Set[str]], min_genes: int = 5
) -> pd.DataFrame:
    """
    Run enrichment analysis for all modules against all pathways.
    """
    background = set(modules_df["Gene"])
    results = []

    for module_id in sorted(modules_df["Module"].unique()):
        if module_id == 0:  # Skip unassigned
            continue

        module_genes = set(modules_df[modules_df["Module"] == module_id]["Gene"])

        if len(module_genes) < min_genes:
            continue

        for pathway_name, pathway_genes in pathway_sets.items():
            enrichment = fisher_enrichment(module_genes, pathway_genes, background)

            results.append(
                {
                    "Module": module_id,
                    "Pathway": pathway_name,
                    "Overlap": enrichment["overlap_count"],
                    "ModuleSize": enrichment["module_size"],
                    "PathwaySize": enrichment["pathway_size"],
                    "OddsRatio": enrichment["odds_ratio"],
                    "PValue": enrichment["pvalue"],
                    "OverlapGenes": ", ".join(sorted(enrichment["overlap_genes"])),
                }
            )

    results_df = pd.DataFrame(results)

    # Adjust p-values (Benjamini-Hochberg)
    if len(results_df) > 0:
        from scipy.stats import false_discovery_control

        try:
            results_df["AdjPValue"] = false_discovery_control(results_df["PValue"])
        except:
            # Fallback: simple Bonferroni correction
            results_df["AdjPValue"] = results_df["PValue"] * len(results_df)
            results_df["AdjPValue"] = results_df["AdjPValue"].clip(upper=1.0)

    return results_df.sort_values("PValue")


enrichment_results = run_enrichment_analysis(
    modules, PATHWAY_GENE_SETS, min_genes=CONFIG.min_genes_for_enrichment
)

print(f"Enrichment analysis complete: {len(enrichment_results)} tests performed")
enrichment_results.head(15)

Enrichment analysis complete: 20 tests performed


,Module,Pathway,Overlap,ModuleSize,PathwaySize,OddsRatio,PValue,OverlapGenes,AdjPValue
6,2,Cell_Cycle,20,20,20,inf,1.725215e-17,"AURKA, AURKB, BUB1, CCNB1, CCND1, CCNE1, CDC25...",3.450430e-16
12,3,DNA_Damage_Response,20,20,21,inf,3.622951e-16,"ATM, ATR, BRCA1, BRCA2, ERCC1, ERCC2, FEN1, LI...",3.622951e-15
0,1,Apoptosis,18,20,18,inf,2.028853e-14,"AIF, APAF1, BAX, BCL2, BCL2L1, BID, BIRC5, CAS...",1.352568e-13
19,4,Metabolism,7,7,7,inf,1.149890e-09,"AMPK, CPT1A, GLS2, GLUT1, HK2, SIRT3, TIGAR",5.749451e-09
3,1,p53_Signaling,5,20,7,7.500000,2.145588e-02,"BAX, MDM2, NOXA, PUMA, TP53",8.582350e-02
18,4,p53_Signaling,1,7,7,1.500000,5.559044e-01,TIGAR,1.000000e+00
8,2,p53_Signaling,1,20,7,0.359649,9.276817e-01,CDKN1A,1.000000e+00
2,1,DNA_Damage_Response,1,20,21,0.071053,9.999032e-01,PARP1,1.000000e+00
7,2,DNA_Damage_Response,0,20,21,0.000000,1.000000e+00,,1.000000e+00
5,2,Apoptosis,0,20,18,0.000000,1.000000e+00,,1.000000e+00


In [9]:
# Show significant enrichments
significant = enrichment_results[enrichment_results["PValue"] < CONFIG.pvalue_threshold]
print(f"\nSignificant enrichments (p < {CONFIG.pvalue_threshold}): {len(significant)}")
significant


Significant enrichments (p < 0.05): 5


,Module,Pathway,Overlap,ModuleSize,PathwaySize,OddsRatio,PValue,OverlapGenes,AdjPValue
6,2,Cell_Cycle,20,20,20,inf,1.725215e-17,"AURKA, AURKB, BUB1, CCNB1, CCND1, CCNE1, CDC25...",3.450430e-16
12,3,DNA_Damage_Response,20,20,21,inf,3.622951e-16,"ATM, ATR, BRCA1, BRCA2, ERCC1, ERCC2, FEN1, LI...",3.622951e-15
0,1,Apoptosis,18,20,18,inf,2.028853e-14,"AIF, APAF1, BAX, BCL2, BCL2L1, BID, BIRC5, CAS...",1.352568e-13
19,4,Metabolism,7,7,7,inf,1.149890e-09,"AMPK, CPT1A, GLS2, GLUT1, HK2, SIRT3, TIGAR",5.749451e-09
3,1,p53_Signaling,5,20,7,7.5,2.145588e-02,"BAX, MDM2, NOXA, PUMA, TP53",8.582350e-02


## 6. Module Characterization
Summarize each module's biological function based on enrichment results.

In [10]:
def characterize_modules(
    modules_df: pd.DataFrame,
    enrichment_df: pd.DataFrame,
    hubs_df: pd.DataFrame,
    pvalue_threshold: float = 0.05,
) -> pd.DataFrame:
    """
    Create a summary characterization of each module.
    """
    summaries = []

    for module_id in sorted(modules_df["Module"].unique()):
        if module_id == 0:
            continue

        # Module genes
        module_genes = modules_df[modules_df["Module"] == module_id]["Gene"].tolist()

        # Hub genes in this module
        module_hubs = hubs_df[
            (hubs_df["Gene"].isin(module_genes)) & (hubs_df["IsHub"])
        ]["Gene"].tolist()

        # Significant pathways
        sig_pathways = enrichment_df[
            (enrichment_df["Module"] == module_id)
            & (enrichment_df["PValue"] < pvalue_threshold)
        ]["Pathway"].tolist()

        # Top pathway
        module_enrichments = enrichment_df[enrichment_df["Module"] == module_id]
        if len(module_enrichments) > 0:
            top_pathway = module_enrichments.iloc[0]["Pathway"]
            top_pvalue = module_enrichments.iloc[0]["PValue"]
        else:
            top_pathway = "None"
            top_pvalue = 1.0

        summaries.append(
            {
                "Module": module_id,
                "Size": len(module_genes),
                "NumHubs": len(module_hubs),
                "HubGenes": ", ".join(module_hubs[:5]),  # Top 5 hubs
                "TopPathway": top_pathway,
                "TopPValue": top_pvalue,
                "SignificantPathways": ", ".join(sig_pathways),
            }
        )

    return pd.DataFrame(summaries)


module_summary = characterize_modules(
    modules, enrichment_results, hubs, CONFIG.pvalue_threshold
)
module_summary

,Module,Size,NumHubs,HubGenes,TopPathway,TopPValue,SignificantPathways
0,1,20,2,"APAF1, NOXA",Apoptosis,2.028853e-14,"Apoptosis, p53_Signaling"
1,2,20,2,"CDKN1A, E2F2",Cell_Cycle,1.725215e-17,Cell_Cycle
2,3,20,2,"XRCC1, ERCC2",DNA_Damage_Response,3.622951e-16,DNA_Damage_Response
3,4,7,1,AMPK,Metabolism,1.149890e-09,Metabolism


## 7. TP53-Specific Analysis
Analyze TP53's position in the network and its module associations.

In [11]:
def analyze_tp53(modules_df: pd.DataFrame, hubs_df: pd.DataFrame) -> Dict:
    """
    Analyze TP53's role in the network.
    """
    # Find TP53
    tp53_data = hubs_df[hubs_df["Gene"] == "TP53"]

    if len(tp53_data) == 0:
        return {"error": "TP53 not found in dataset"}

    tp53_row = tp53_data.iloc[0]
    tp53_module = tp53_row["Module"]

    # Get co-module genes
    co_module_genes = modules_df[modules_df["Module"] == tp53_module]["Gene"].tolist()
    co_module_hubs = hubs_df[
        (hubs_df["Gene"].isin(co_module_genes)) & (hubs_df["IsHub"])
    ]["Gene"].tolist()

    return {
        "gene": "TP53",
        "module": tp53_module,
        "is_hub": tp53_row["IsHub"],
        "connectivity": tp53_row["Connectivity"],
        "intramodular_connectivity": tp53_row["IntramodularConnectivity"],
        "module_size": len(co_module_genes),
        "module_hubs": co_module_hubs,
        "module_genes_sample": co_module_genes[:10],
    }


tp53_analysis = analyze_tp53(modules, hubs)
print("\nTP53 Network Analysis:")
for key, value in tp53_analysis.items():
    print(f"  {key}: {value}")


TP53 Network Analysis:
  gene: TP53
  module: 1
  is_hub: False
  connectivity: 23.742403473504723
  intramodular_connectivity: 13.708453177673428
  module_size: 20
  module_hubs: ['APAF1', 'NOXA']
  module_genes_sample: ['BIRC5', 'APAF1', 'HTRA2', 'CASP9', 'MDM2', 'BAX', 'CASP3', 'AIF', 'XIAP', 'MCL1']


## 8. Biological Interpretation
Generate a structured interpretation of the results.

In [12]:
def generate_interpretation(
    module_summary: pd.DataFrame, enrichment_results: pd.DataFrame, tp53_analysis: Dict
) -> Dict:
    """
    Generate structured biological interpretation.
    """
    interpretation = {
        "network_overview": {
            "total_modules": len(module_summary),
            "total_hub_genes": module_summary["NumHubs"].sum(),
            "modules_with_significant_enrichment": len(
                module_summary[module_summary["TopPValue"] < 0.05]
            ),
        },
        "tp53_findings": {
            "module": tp53_analysis.get("module", "Unknown"),
            "is_hub": tp53_analysis.get("is_hub", False),
            "connectivity_rank": "High"
            if tp53_analysis.get("is_hub", False)
            else "Moderate",
        },
        "key_modules": [],
    }

    # Characterize key modules
    for _, row in module_summary.iterrows():
        if row["TopPValue"] < 0.05:
            interpretation["key_modules"].append(
                {
                    "module_id": row["Module"],
                    "size": row["Size"],
                    "primary_function": row["TopPathway"],
                    "hub_genes": row["HubGenes"],
                }
            )

    return interpretation


interpretation = generate_interpretation(
    module_summary, enrichment_results, tp53_analysis
)

print("\n" + "=" * 60)
print("BIOLOGICAL INTERPRETATION SUMMARY")
print("=" * 60)

print(f"\nNetwork Overview:")
for key, value in interpretation["network_overview"].items():
    print(f"  - {key.replace('_', ' ').title()}: {value}")

print(f"\nTP53 Findings:")
for key, value in interpretation["tp53_findings"].items():
    print(f"  - {key.replace('_', ' ').title()}: {value}")

print(f"\nKey Modules:")
for mod in interpretation["key_modules"]:
    print(f"  Module {mod['module_id']}: {mod['primary_function']}")
    print(f"    Size: {mod['size']}, Hub genes: {mod['hub_genes']}")


BIOLOGICAL INTERPRETATION SUMMARY

Network Overview:
  - Total Modules: 4
  - Total Hub Genes: 7
  - Modules With Significant Enrichment: 4

TP53 Findings:
  - Module: 1
  - Is Hub: False
  - Connectivity Rank: Moderate

Key Modules:
  Module 1: Apoptosis
    Size: 20, Hub genes: APAF1, NOXA
  Module 2: Cell_Cycle
    Size: 20, Hub genes: CDKN1A, E2F2
  Module 3: DNA_Damage_Response
    Size: 20, Hub genes: XRCC1, ERCC2
  Module 4: Metabolism
    Size: 7, Hub genes: AMPK


## 9. Export Results
Save enrichment results and interpretations.

In [13]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save enrichment results
enrichment_results.to_csv(EXPORT_DIR / "task5_enrichment_results.csv", index=False)
print(
    f"[OK] Enrichment results saved to: {EXPORT_DIR / 'task5_enrichment_results.csv'}"
)

# Save module summary
module_summary.to_csv(EXPORT_DIR / "task5_module_summary.csv", index=False)
print(f"[OK] Module summary saved to: {EXPORT_DIR / 'task5_module_summary.csv'}")

# Save significant enrichments only
significant.to_csv(EXPORT_DIR / "task5_significant_enrichments.csv", index=False)
print(
    f"[OK] Significant enrichments saved to: {EXPORT_DIR / 'task5_significant_enrichments.csv'}"
)

# Save interpretation as JSON
import json


def convert_to_serializable(obj):
    """Convert numpy types to Python native types for JSON serialization."""
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    return obj


with open(EXPORT_DIR / "task5_interpretation.json", "w") as f:
    json.dump(convert_to_serializable(interpretation), f, indent=2)
print(f"[OK] Interpretation saved to: {EXPORT_DIR / 'task5_interpretation.json'}")

# Save TP53 analysis
pd.DataFrame([tp53_analysis]).to_csv(
    EXPORT_DIR / "task5_tp53_analysis.csv", index=False
)
print(f"[OK] TP53 analysis saved to: {EXPORT_DIR / 'task5_tp53_analysis.csv'}")

print("\n[OK] All enrichment analysis results exported successfully!")

[OK] Enrichment results saved to: artifacts/task5_enrichment_results.csv
[OK] Module summary saved to: artifacts/task5_module_summary.csv
[OK] Significant enrichments saved to: artifacts/task5_significant_enrichments.csv
[OK] Interpretation saved to: artifacts/task5_interpretation.json
[OK] TP53 analysis saved to: artifacts/task5_tp53_analysis.csv

[OK] All enrichment analysis results exported successfully!
